# 流特征提取

In [10]:
import os
from scapy.utils import rdpcap
from scapy.layers.inet import IP, TCP
import torch  # For tensor storage
from scapy.all import *
from typing import Dict, Tuple

## 前置处理
根据 summary.txt 获取有效的五元组: `(src_ip, src_port, dst_ip, dst_port)`
为了兼容双向流，可以保证 `src_ip < dst_ip`

In [11]:
# 过滤pcap文件，只提取指定四元组的报文
def build_vaild_flow_ids(summary_file):
    vaild_flow_ids = []
    with open(summary_file, "r") as f:
        lines = f.readlines()
        # 转换为四元组
        for line in lines:
            flow = eval(line)
            src_ip, src_port, dst_ip, dst_port = flow
            flow_id = (src_ip, src_port, dst_ip, dst_port)
            reverse_flow_id = (dst_ip, dst_port, src_ip, src_port)
            vaild_flow_ids.append(min(flow_id, reverse_flow_id))
    return vaild_flow_ids


# print(vaild_ids)

## 流特征提取 & 计算

流的各维度特征计算，目前单流纬度特征包括：
1. 包长序列        packet_length
2. 包负载长度序列   payload_length
3. 包标记位序列     flags
4. 上行/下行       direction
5. 包时间戳序列     timestamp
6. 握手包内容(Server hello)       handshark   
7. 待完善...

In [12]:
import pandas as pd


# Helper function to extract flow identifier
def get_flow_id(packet):
    ip_layer = packet[IP]
    tcp_layer = packet[TCP]
    return (ip_layer.src, tcp_layer.sport, ip_layer.dst, tcp_layer.dport)


# 获取双向流的ID
def get_bidirectional_flow_id(packet):
    ip_layer = packet[IP]
    tcp_layer = packet[TCP]
    # Create a flow identifier
    flow_id = (ip_layer.src, tcp_layer.sport, ip_layer.dst, tcp_layer.dport)
    reverse_flow_id = (ip_layer.dst, tcp_layer.dport, ip_layer.src, tcp_layer.sport)
    # Return the lexicographically smaller tuple to ensure consistency
    return min(flow_id, reverse_flow_id)


# Helper function to extract packet length
def get_packet_length(packet):
    return len(packet)


# Helper function to extract packet timestamp
def get_packet_timestamp(packet):
    return float(packet.time)


# Helper function to extract TCP flags
def get_tcp_flags(packet):
    return hex(int(packet[TCP].flags))


# Helper function to extract Payload Len
def get_payload_length(packet):
    if TCP in packet:
        # Get the payload of the TCP layer
        payload = packet[TCP].payload
        # Return the length of the payload
        return len(payload)
    return 0  # Return 0 if no payload exists


# Helper function to determine if a packet is uplink or downlink
def is_uplink(packet, flow_id):
    """
    Determine if a packet is uplink (client to server) or downlink (server to client).

    Args:
        packet: A Scapy packet object.
        flow_id: A tuple (src_ip, src_port, dst_ip, dst_port) representing the flow.

    Returns:
        str: "uplink" if the packet is client to server, "downlink" if server to client.
    """
    src_ip, src_port, dst_ip, dst_port = flow_id

    # Check if the packet matches the uplink direction
    if packet[IP].src == src_ip and packet[TCP].sport == src_port:
        return "uplink"  # Client to server

    # Check if the packet matches the downlink direction
    if packet[IP].src == dst_ip and packet[TCP].sport == dst_port:
        return "downlink"  # Server to client

    return "unknown"  # If it doesn't match either direction


# 提取流的 握手包内容, 需要具体内容
def extract_handshake_payload(packet) -> Tuple[Dict, bool]:
    # Read packets from the pcap file
    tls_data = {}
    vaild_handshake = False
    if not packet.haslayer("SSL/TLS"):
        return tls_data, False
    tls = packet.getlayer("SSL/TLS")
    if "TLS Record" not in tls:
        # print(f"No TLS Record found in packet: {packet.summary()}")
        return {}, False
    tls_data["tls_vers"] = hex(tls["TLS Record"].version)  # 0x303: TLS 1.2
    tls_data["tls_len"] = hex(tls["TLS Record"].length)
    if "TLS Handshake" in tls:
        tls_data["tls_step"] = tls["TLS Handshake"].type
        if tls_data["tls_step"] == 2:  # Server Hello
            # tls server hello length
            tls_data["tls_shlen"] = tls["TLS Handshake"].length
            # tls cipher ECDHE_RSA_WITH_AES_128_CBC_SHA256 0xc027 -> 49191
            tls_data["tls_cip"] = tls["TLS Handshake"].cipher_suite
            # tls compression method 0x00 -> 0
            tls_data["tls_comp"] = tls["TLS Handshake"].compression_method
            # tls extensions length 0x8 -> 8
            tls_data["tls_extlen"] = "0x" + str(tls["TLS Handshake"].extensions_length)
            # tls extensions type 0x000b -> 11
            tls_data["tls_exttype"] = tls["TLS Extension"].type
            vaild_handshake = True
        elif tls_data["tls_step"] == 11:  # Certificate Message
            tls.show()
            # tls_data['tls_certificate'] = tls['TLS Handshake']
        elif tls_data["tls_step"] == 12:  # Server Key Exchange todo
            tls.show()
        elif tls_data["tls_step"] == 14:  # Server Hello Done todo
            tls.show()
    return tls_data, vaild_handshake


# 提取流信息的函数
def extract_flows(
    pcap_file: str, extract_features: list, vaild_flow_ids=None
) -> Dict[str, Dict]:
    """
    Extract packet length sequences for each TCP flow from a pcap file.

    Args:
        pcap_file (str): Path to the pcap file.
        extract_features (list): List of features to extract from packets.

    Returns:
        dict: A dictionary where keys are flow identifiers (e.g., tuple of IPs and ports)
              and values are lists of packet lengths.
    """
    if not os.path.exists(pcap_file):
        raise FileNotFoundError(f"PCAP file not found: {pcap_file}")

    # Read packets from the pcap file
    packets = rdpcap(pcap_file)

    # 序列特征
    flows = {}  # Dictionary to store flows and their packet lengths
    # 切分成流, 流纬度特征提取
    for packet in packets:
        # Check if the packet has IP and TCP layers
        if IP in packet and TCP in packet:
            # print(f"Processing packet in flow: {bi_flow_id}")
            # flow_id = get_flow_id(packet) # 单向流
            flow_id = get_bidirectional_flow_id(packet)  # 双向流

            # 过滤背景流
            if vaild_flow_ids != None and flow_id not in vaild_flow_ids:
                continue
            raw_flow_id = tuple(flow_id)
            flow_id = str(flow_id)
            if flow_id not in flows:
                flows[flow_id] = {}
            if "flow_start_time" in extract_features:
                if "flow_start_time" not in flows[flow_id]:
                    flows[flow_id]["flow_start_time"] = get_packet_timestamp(packet)
            if "packet_length" in extract_features:
                flows[flow_id].setdefault("packet_length", []).append(
                    get_packet_length(packet)
                )
            if "timestamp" in extract_features:
                flows[flow_id].setdefault("timestamp", []).append(
                    get_packet_timestamp(packet)
                )
            if "flags" in extract_features:
                flows[flow_id].setdefault("flags", []).append(get_tcp_flags(packet))
            if "handshake" in extract_features:
                handshake_payload, vaild = extract_handshake_payload(packet)
                if vaild:
                    flows[flow_id].setdefault("handshake", []).append(handshake_payload)
            if "payload_length" in extract_features:
                flows[flow_id].setdefault("payload_length", []).append(
                    get_payload_length(packet)
                )
            if "direction" in extract_features:
                flows[flow_id].setdefault("direction", []).append(
                    is_uplink(packet, raw_flow_id)
                )
    # 包长统计特征计算
    for flow_id in flows:
        packet_lenghts = flows[flow_id].get("packet_length", [])
        # 截取 60 个包
        flows[flow_id]["packet_length"] = packet_lenghts[:60]
        # 转换成 pandas Series 计算统计特征
        flows_series = pd.Series(packet_lenghts)
        # minimum, maximum, mean, median absolute deviation, standard deviation, variance,
        # skew, kurtosis, percentiles (from 10% to 90%), and the number of elements in the series (18 in total).
        flows[flow_id]["length_min"] = flows_series.min()
        flows[flow_id]["length_max"] = flows_series.max()
        flows[flow_id]["length_mean"] = flows_series.mean()
        flows[flow_id]["length_mad"] = flows_series.mad()
        flows[flow_id]["length_std"] = flows_series.std()
        flows[flow_id]["length_var"] = flows_series.var()
        flows[flow_id]["length_skew"] = flows_series.skew()
        flows[flow_id]["length_kurt"] = flows_series.kurt()
        for perc in range(10, 100, 10):
            flows[flow_id][f"length_perc_{perc}"] = flows_series.quantile(perc / 100.0)
        flows[flow_id]["length_count"] = flows_series.count()

    return flows

## 特征存储

特征的存储，目前支持的存储格式为：

1. Json格式
2. Tensor格式

In [13]:
import json
import pandas as pd
import numpy as np
from typing import List, Dict


# 自定义 JSON 编码器，处理 numpy 数据类型
class NumpyEncoder(json.JSONEncoder):
    """自定义 JSONEncoder，将 numpy 类型转换为 Python 原生类型"""

    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


# 保存成 tensor 张量 todo
def sink_tensors_file(flows, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # TLS Layer Feature
    """
        "tls_vers":"0x303",
        "tls_len":"0x7a",
        "tls_step":2,
        "tls_shlen":118,
        "tls_cip":4865,
        "tls_comp":0,
        "tls_extlen":46,
        "tls_exttype":43
    """

    tls_records: List[Dict] = []

    for i, (flow_id, feature) in enumerate(flows.items()):
        # Convert packet info to a PyTorch tensor
        handshake_payload = feature.get("handshake", [])
        if len(handshake_payload) > 0:
            tls_records.append(handshake_payload[0])

    fields = [
        "tls_vers",
        "tls_len",
        "tls_step",
        "tls_shlen",
        "tls_cip",
        "tls_comp",
        "tls_extlen",
        "tls_exttype",
    ]
    df = pd.DataFrame(tls_records)
    df = df.reindex(columns=fields)  # 保证列顺序并只留需要字段
    df = df.fillna("")  # 用默认值填充
    # 转为 numpy 字符串矩阵或做后续编码
    X_str = df.to_numpy(dtype=object)
    print(X_str)


def sink_json_file(flows, output_dir, idx: str):
    """
    Save flows as JSON files to the specified directory.

    Args:
        flows (dict): A dictionary where keys are flow identifiers and values are lists of packet info dicts.
        output_dir (str): Directory to save the JSON files.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    json.dump(
        flows,
        open(os.path.join(output_dir, "feature.json"), "w"),
        cls=NumpyEncoder,
        indent=4,
        separators=(",", ":"),
    )

In [14]:
# Encoding TLS features: one-hot for certain fields, others to int and save as matrix
import os
import numpy as np
import joblib
import pandas as pd
from sklearn.preprocessing import OneHotEncoder


def encode_tls_features_and_save(flows, output_dir, instance_id: str):
    """
    从 flows 中提取 TLS 握手记录，对指定字段做 one-hot 编码，其余数值字段转成 int，
    最终得到一个 numpy 矩阵并保存，同时保存 OneHotEncoder 对象以便推理时复用。
    参数：
        flows: dict, extract_flows 返回的 flows 结构
        output_dir: str, 保存输出的目录（会创建）
    返回： (X, ohe, fields_order)
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    # 期望字段顺序（与 sink_tensors_file 保持一致）
    fields = [
        "tls_vers",
        "tls_len",
        "tls_step",
        "tls_shlen",
        "tls_cip",
        "tls_comp",
        "tls_extlen",
        "tls_exttype",
    ]

    # 收集每条 flow 的第一个 handshake 字典（若存在）
    records = []
    for fid, feat in flows.items():
        hs = feat.get("handshake", [])
        if len(hs) > 0 and isinstance(hs[0], dict):
            records.append(hs[0])

    df = pd.DataFrame(records)
    # 保证列顺序并补空
    df = df.reindex(columns=fields)
    df = df.fillna("")

    # 把 hex 字符串(如 '0x7a') 或字符串数字 转为 int，对于不能转换的填 0
    def hex_to_int_safe(x):
        if x is None or x == "":
            return 0
        if isinstance(x, (int, float)):
            try:
                return int(x)
            except Exception:
                return 0
        s = str(x)
        if s.startswith("0x") or s.startswith("0X"):
            try:
                return int(s, 16)
            except Exception:
                return 0
        try:
            return int(float(s))
        except Exception:
            return 0

    # 需要 one-hot 的列
    onehot_cols = ["tls_vers", "tls_step", "tls_cip", "tls_comp", "tls_exttype"]
    # 需要作为数值 int 的列
    int_cols = ["tls_len", "tls_shlen", "tls_extlen"]

    # 先把 int_cols 转换为 int 数值
    for c in int_cols:
        if c in df.columns:
            df[c] = df[c].apply(hex_to_int_safe)
        else:
            df[c] = 0

    # 对 one-hot 列，先转为字符串（保证分类一致），缺失用特殊字符串 '__MISSING__'
    for c in onehot_cols:
        if c not in df.columns:
            df[c] = "__MISSING__"
        else:
            df[c] = df[c].apply(
                lambda x: "__MISSING__" if x is None or x == "" else str(x)
            )

    # fit OneHotEncoder: 兼容不同 sklearn 版本的参数名称
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    ohe_arr = ohe.fit_transform(df[onehot_cols])  # shape (n_samples, n_ohe_features)

    # 构建 int array
    int_arr = df[int_cols].to_numpy(dtype=np.int32)  # shape (n_samples, len(int_cols))

    # 合并为最终矩阵（one-hot 在前，int 在后）
    X = np.concatenate([ohe_arr, int_arr], axis=1) if int_arr.size else ohe_arr
    np.set_printoptions(suppress=True, precision=4, threshold=100000)
    # 保存结果与 encoder
    np.save(os.path.join(output_dir, f"X_tls_{instance_id}.npy"), X)
    joblib.dump(ohe, os.path.join(output_dir, f"ohe_tls.joblib"))

    # 记录列顺序信息便于推理时恢复（onehot 输出顺序 + int_cols）
    try:
        ohe_feature_names = ohe.get_feature_names_out(onehot_cols).tolist()
    except Exception:
        try:
            ohe_feature_names = list(ohe.get_feature_names(onehot_cols))
        except Exception:
            ohe_feature_names = []
            if hasattr(ohe, "categories_"):
                for i, c in enumerate(onehot_cols):
                    cats = ohe.categories_[i]
                    ohe_feature_names += [f"{c}__{v}" for v in cats]
    fields_order = ohe_feature_names + int_cols

    joblib.dump(fields_order, os.path.join(output_dir, "fields_order.joblib"))
    print(f"TLS feature fields order: {fields_order}")
    print("Saved X_tls.npy shape=", X.shape)
    return X, ohe, fields_order

## 全链路链路测试

执行前置处理、特征提取、特征计算、全链路测试

In [16]:
import os

# 特征提取主流程
root_dir = "/home/tyf/Project/encrypt_traffic/train_raw_data"
remote_root_dir = "/home/tyf/fnnas/Study/Traffic-data/train_raw_data"
idx = 1

web_instance_counter = {}
failed_instances = []


def draw_instance_counts(web_instance_counter):
    """绘制每个网站实例数量的柱状图"""
    import matplotlib.pyplot as plt

    websites = list(web_instance_counter.keys())
    counts = list(web_instance_counter.values())
    plt.figure(figsize=(12, 6))
    plt.bar(websites, counts)
    plt.xlabel("Website")
    plt.ylabel("Number of Instances")
    plt.title("Number of Instances per Website")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("./results/website_instance_counts.png")


website_idx = 0
for website_name in os.listdir(remote_root_dir):
    website_idx = website_idx + 1
    website_folder = os.path.join(remote_root_dir, website_name)
    if not os.path.isdir(website_folder):
        continue

    for instance_id in os.listdir(website_folder):
        data_dir = os.path.join(website_folder, instance_id)
        if not os.path.isdir(data_dir):
            continue

        web_instance_counter[website_name] = (
            web_instance_counter.get(website_name, 0) + 1
        )

        # Find pcap file and summary file
        pcap_file = None
        summary_file = None
        for filename in os.listdir(data_dir):
            if filename == "traffic.pcap":
                pcap_file = os.path.join(data_dir, filename)
            elif filename == "summary.txt":
                summary_file = os.path.join(data_dir, filename)

        if pcap_file is None or summary_file is None:
            failed_instances.append(data_dir)
            continue

        # Build valid flow IDs
        vaild_flow_ids = build_vaild_flow_ids(summary_file)

        # Define output directory
        dst_feature_dir = f"../raw_feature/{website_name}"
        output_dir = dst_feature_dir

        print(f"Processing website {website_name}, instance {instance_id}")

        # Extract packet lengths for each TCP flow
        flows = extract_flows(
            pcap_file,
            extract_features=[
                "packet_length",
                "payload_length",
                "direction",
                "timestamp",
                "flow_start_time",
                "flags",
                "handshake",
            ],
            vaild_flow_ids=vaild_flow_ids,
        )

        # Save flows
        sink_json_file(flows, output_dir, f"{website_idx}-{instance_id}")

        # sink_tensors_file(flows, output_dir)
        # encode_tls_features_and_save(flows, output_dir, f"{website_idx}-{instance_id}")
        idx = idx + 1
        if idx % 100 == 0:
            print(f"Processed instance {idx} for website {website_name}")

draw_instance_counts(web_instance_counter)
print("Failed instances:", failed_instances)

Processing website blog.csdn, instance eda87b35-c8ab-4fb3-986e-cbbbe7b2907f
Processing website blog.csdn, instance 80a34d31-ffb5-4368-bb5c-af275f28a561
Processing website blog.csdn, instance 1473763d-130f-4d82-b79b-7e74c5f29173


KeyboardInterrupt: 